# CARLA 0.9.15 on Kaggle: smoke test

Checks whether CARLA can run headless on a free Kaggle GPU, and whether it has enough memory for the
recordings that crashed on the laptop (runtime switch from Town10 to Town03 with 40 vehicles, 60 walkers
and three 800x600 cameras).

**Before running** (right-hand panel, *Session options*):
- Accelerator: **GPU T4 x1** (or P100)
- Internet: **On** (needs a phone-verified Kaggle account)

Then **Run All**. It takes about 20-40 minutes, mostly the CARLA download. The last cell prints a verdict and
a JSON report; paste that output back. The test frames and `report.json` land in `/kaggle/working/carla_test`.

Steps: environment report -> system packages -> Vulkan (CARLA renders through Vulkan even headless) ->
CARLA download -> Python 3.10 client env (the `carla` 0.9.15 wheel has no build for newer Pythons) ->
start the server as a non-root user (CARLA refuses root) -> record in two towns -> verdict.

In [ ]:
import glob
import json
import os
import shutil
import socket
import subprocess
import time


def sh(cmd, check=True, quiet=False):
    """Run a bash command, stream its output, return (returncode, last output lines)."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(
        ["bash", "-o", "pipefail", "-c", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    lines = []
    try:
        for line in proc.stdout:
            lines.append(line)
            if not quiet:
                print(line, end="", flush=True)
        proc.wait()
    except KeyboardInterrupt:  # Stop button: don't leave the command running in the background
        proc.kill()
        raise
    if check and proc.returncode != 0:
        raise RuntimeError(f"exit {proc.returncode}: {cmd}\n{''.join(lines[-30:])}")
    return proc.returncode, "".join(lines[-400:])


REPORT = {}
sh("nvidia-smi")
sh("free -g; echo cpus: $(nproc); grep PRETTY_NAME /etc/os-release; python --version; whoami")
_, gpu = sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader")
REPORT["gpu"] = gpu.strip()
DRIVER = gpu.strip().split(", ")[1]
REPORT["ram_total_gb"] = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1)

candidates = [d for d in ("/tmp", "/kaggle/tmp", "/root") if os.path.isdir(d)]
free = {d: round(shutil.disk_usage(d).free / 1e9, 1) for d in candidates}
WORK = max(free, key=free.get)
REPORT["disk_free_gb"] = free
print("free disk (GB):", free, "-> working in", WORK)
assert free[WORK] > 30, "need ~30 GB free for CARLA"
CARLA_DIR = f"{WORK}/carla"
OUT = "/kaggle/working/carla_test" if os.path.isdir("/kaggle/working") else f"{WORK}/carla_test"
os.makedirs(OUT, exist_ok=True)

## System packages

In [ ]:
apt = (
    "DEBIAN_FRONTEND=noninteractive timeout 600 apt-get install -y --no-install-recommends "
    "-o DPkg::Lock::Timeout=120"
)
sh("timeout 300 apt-get update -qq", check=False)
sh(f"{apt} libvulkan1 vulkan-tools xdg-user-dirs")
if sh(f"{apt} libomp5", check=False)[0] != 0:  # Ubuntu 24.04 names it libomp5-<llvm version>
    sh(f"{apt} libomp5-18")
for pkg in ("libsdl2-2.0-0", "libjpeg8", "libpng16-16t64", "fontconfig", "libtiff6"):
    sh(f"{apt} {pkg}", check=False)  # nice-to-have
print("packages installed")

## Vulkan

Containers often get the NVIDIA compute libraries but not the graphics ones. If `libGLX_nvidia` (the NVIDIA
Vulkan driver) is missing, extract it from the NVIDIA driver installer of the exact same version and register
it; nothing is installed into the kernel.

In [ ]:
# The container mounts the host driver's libraries, often only the compute ones: libGLX_nvidia (the Vulkan
# entry point) is there but its helpers (libnvidia-glvkspirv, -gpucomp, -glcore, ...) are not, and Vulkan
# fails with "Found no drivers". Fill the gaps from NVIDIA's installer for the exact same driver version
# (files are only extracted; nothing touches the kernel).
apt = (
    "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends -o DPkg::Lock::Timeout=120"
)
sh(f"{apt} libx11-6 libxext6", check=False)
SYS_DIR = "/usr/local/nvidia/lib64"
sh(f"ls {SYS_DIR} | grep -Ei 'glx|glcore|glvk|gpucomp|tls|egl' || true")
sh(f"ldd {SYS_DIR}/libGLX_nvidia.so.0 | grep 'not found' || echo 'no missing link-time deps'", check=False)
REPORT["glx_nvidia_in_container"] = os.path.exists(f"{SYS_DIR}/libGLX_nvidia.so.0")

run_file, drv_dir = f"{WORK}/nvidia.run", f"{WORK}/nvdrv"
if not os.path.isdir(drv_dir):
    urls = [
        f"https://us.download.nvidia.com/tesla/{DRIVER}/NVIDIA-Linux-x86_64-{DRIVER}.run",
        f"https://us.download.nvidia.com/XFree86/Linux-x86_64/{DRIVER}/NVIDIA-Linux-x86_64-{DRIVER}.run",
        f"https://download.nvidia.com/XFree86/Linux-x86_64/{DRIVER}/NVIDIA-Linux-x86_64-{DRIVER}.run",
    ]
    for url in urls:
        if sh(f"curl -fsSL --retry 3 -o {run_file} {url}", check=False)[0] == 0:
            print("got", url)
            break
    else:
        raise RuntimeError(f"no NVIDIA installer found for driver {DRIVER}")
    sh(f"chmod +x {run_file}")
    sh(f"{run_file} --extract-only --target {drv_dir}", quiet=True)

VK_DIR = "/usr/local/nvidia-vk"
os.makedirs(VK_DIR, exist_ok=True)
glx = f"{VK_DIR}/libGLX_nvidia.so.{DRIVER}"
shutil.copy(f"{drv_dir}/libGLX_nvidia.so.{DRIVER}", glx)
copied = []
for lib in glob.glob(f"{drv_dir}/libnvidia-*.so.{DRIVER}"):
    name = os.path.basename(lib)
    if not os.path.exists(f"{SYS_DIR}/{name}"):  # keep the host's own copies of what is mounted
        shutil.copy(lib, VK_DIR)
        copied.append(name)
print("added from installer:", sorted(copied))
with open("/etc/ld.so.conf.d/nvidia-vk.conf", "w") as f:
    f.write(VK_DIR + "\n")
sh("ldconfig")

ICD = "/etc/vulkan/icd.d/nvidia_icd.json"
os.makedirs(os.path.dirname(ICD), exist_ok=True)
with open(ICD, "w") as f:
    json.dump({"file_format_version": "1.0.0", "ICD": {"library_path": glx, "api_version": "1.3.0"}}, f)
os.environ["VK_ICD_FILENAMES"] = ICD

_, out = sh("vulkaninfo --summary", check=False)
REPORT["vulkan_ok"] = "deviceName" in out and "NVIDIA" in out.upper() and "llvmpipe" not in out
REPORT["vulkan_backend"] = "nvidia"
if not REPORT["vulkan_ok"]:
    # Kaggle gives containers compute-only GPU access (NVIDIA_DRIVER_CAPABILITIES=compute,utility, and
    # /dev/nvidia-modeset can't be opened), so fall back to lavapipe, Mesa's CPU Vulkan. Much slower, but sync
    # mode waits for every frame, so recordings come out the same.
    sh(f"{apt} mesa-vulkan-drivers", check=False, quiet=True)
    ICD = glob.glob("/usr/share/vulkan/icd.d/lvp_icd*.json")[0]
    os.environ["VK_ICD_FILENAMES"] = ICD
    _, out = sh("vulkaninfo --summary 2>&1 | grep -E 'deviceName|driverName|ERROR' | head", check=False)
    REPORT["vulkan_ok"] = "llvmpipe" in out
    REPORT["vulkan_backend"] = "lavapipe (CPU)"
print(f"\nVULKAN OK ({REPORT['vulkan_backend']})" if REPORT["vulkan_ok"] else "\nVULKAN NOT WORKING")

## Download CARLA 0.9.15 (Linux)

Streamed straight into `tar`, so the archive never needs its own disk space.

In [ ]:
CARLA_URLS = [
    "https://carla-releases.s3.us-east-005.backblazeb2.com/Linux/CARLA_0.9.15.tar.gz",
    "https://carla-releases.s3.eu-west-3.amazonaws.com/Linux/CARLA_0.9.15.tar.gz",
]
os.makedirs(CARLA_DIR, exist_ok=True)
if not os.path.exists(f"{CARLA_DIR}/CarlaUE4.sh"):
    for url in CARLA_URLS:
        start = time.time()
        if sh(f"curl -fsSL --retry 3 {url} | tar -xz -C {CARLA_DIR}", check=False)[0] == 0:
            REPORT["carla_download_min"] = round((time.time() - start) / 60, 1)
            break
        print("failed:", url)
    else:
        raise RuntimeError("could not download CARLA")
sh(f"du -sh {CARLA_DIR}; ls {CARLA_DIR}")

## Python 3.10 client environment

In [ ]:
PY = f"{WORK}/py310/bin/python"
sh("pip install -q uv", quiet=True)
sh(f"uv python install 3.10 && uv venv -q --allow-existing -p 3.10 {WORK}/py310")
deps = "'numpy<2' 'opencv-python-headless<4.11' psutil"
if sh(f"uv pip install -q -p {PY} carla==0.9.15 {deps}", check=False)[0] != 0:
    wheels = glob.glob(f"{CARLA_DIR}/PythonAPI/carla/dist/carla-0.9.15-cp310*.whl")
    print("PyPI install failed, using bundled wheel:", wheels)
    sh(f"uv pip install -q -p {PY} {wheels[0]} {deps}")
sh(f"{PY} -c 'import carla, cv2, numpy; print(\"carla client OK\", numpy.__version__, cv2.__version__)'")

## Start the CARLA server

In [ ]:
sh("id carla >/dev/null 2>&1 || useradd -m carla")
sh(f"chown -R carla:carla {CARLA_DIR}")
LOG = f"{WORK}/carla_server.log"
sh(f"touch {LOG} && chmod 666 {LOG}")
start_cmd = (
    f"cd {CARLA_DIR} && VK_ICD_FILENAMES={ICD} exec ./CarlaUE4.sh -RenderOffScreen -nosound "
    f"-quality-level=Low -carla-rpc-port=2000 > {LOG} 2>&1"
)
# Detached, with no pipes back to this kernel: sh() would block on a pipe the server inherits.
subprocess.Popen(
    ["su", "carla", "-s", "/bin/bash", "-c", start_cmd],
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)


def port_open(port=2000):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("localhost", port)) == 0


start = time.time()
while time.time() - start < 1800 and not port_open():
    time.sleep(5)
REPORT["server_started"] = port_open()
REPORT["server_start_s"] = round(time.time() - start)
time.sleep(10)  # let it settle before measuring idle memory
sh("free -m; nvidia-smi --query-gpu=memory.used,memory.total --format=csv")
sh(f"tail -n 25 {LOG}", check=False)
print("\nSERVER UP" if REPORT["server_started"] else "\nSERVER DID NOT START (see log above)")

## Recording test

Mirrors `carla_scripts/record_sequence.py` (same sensors, traffic and sync settings) but only checks that frames
render and measures memory: starts in Town10HD_Opt, then loads Town03 at runtime, the step that crashed locally.

In [ ]:
%%writefile /tmp/carla_test.py
import argparse
import json
import queue
import random
import subprocess
import time
from pathlib import Path

import carla
import cv2
import numpy as np
import psutil


def gpu_used_mb():
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True
    ).stdout
    return int(out.split()[0])


def server_alive():
    return any("CarlaUE4" in (p.info["name"] or "") for p in psutil.process_iter(["name"]))


class Peaks:
    def __init__(self):
        self.ram_used_gb = self.carla_rss_gb = self.vram_mb = 0.0

    def sample(self):
        vm = psutil.virtual_memory()
        rss = sum(
            p.info["memory_info"].rss
            for p in psutil.process_iter(["name", "memory_info"])
            if "CarlaUE4" in (p.info["name"] or "")
        )
        self.ram_used_gb = max(self.ram_used_gb, (vm.total - vm.available) / 1e9)
        self.carla_rss_gb = max(self.carla_rss_gb, rss / 1e9)
        self.vram_mb = max(self.vram_mb, gpu_used_mb())


def to_bgr(image):
    return np.frombuffer(image.raw_data, np.uint8).reshape(image.height, image.width, 4)[:, :, :3].copy()


def run_town(client, town, args, peaks):
    result = {"town": town}
    start = time.time()
    world = client.load_world(town)
    result["load_s"] = round(time.time() - start, 1)
    peaks.sample()

    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.05
    world.apply_settings(settings)
    tm = client.get_trafficmanager(8000)
    tm.set_synchronous_mode(True)
    tm.set_random_device_seed(0)
    world.set_pedestrians_seed(0)
    rng = random.Random(0)
    lib = world.get_blueprint_library()

    vehicles, walkers, controllers, cameras = [], [], [], []
    try:
        spawn_points = world.get_map().get_spawn_points()
        rng.shuffle(spawn_points)
        ego = world.spawn_actor(lib.find("vehicle.tesla.model3"), spawn_points[0])
        ego.set_autopilot(True, 8000)
        vehicles.append(ego)
        for transform in spawn_points[1:]:
            if len(vehicles) > args.vehicles:
                break
            vehicle = world.try_spawn_actor(rng.choice(lib.filter("vehicle.*")), transform)
            if vehicle is not None:
                vehicle.set_autopilot(True, 8000)
                vehicles.append(vehicle)
        for _ in range(args.walkers * 2):
            if len(walkers) == args.walkers:
                break
            location = world.get_random_location_from_navigation()
            if location is not None:
                walker = world.try_spawn_actor(
                    rng.choice(lib.filter("walker.pedestrian.*")), carla.Transform(location + carla.Location(z=1.0))
                )
                if walker is not None:
                    walkers.append(walker)
        controller_bp = lib.find("controller.ai.walker")
        controllers = [world.spawn_actor(controller_bp, carla.Transform(), attach_to=w) for w in walkers]
        world.tick()
        for controller in controllers:
            controller.start()
            controller.go_to_location(world.get_random_location_from_navigation())
        mount = carla.Transform(carla.Location(x=1.5, z=1.6))
        queues = []
        for blueprint in ("sensor.camera.rgb", "sensor.camera.instance_segmentation", "sensor.camera.depth"):
            bp = lib.find(blueprint)
            bp.set_attribute("image_size_x", "800")
            bp.set_attribute("image_size_y", "600")
            bp.set_attribute("fov", "90")
            camera = world.spawn_actor(bp, mount, attach_to=ego)
            q = queue.Queue()
            camera.listen(q.put)
            cameras.append(camera)
            queues.append(q)
        result["vehicles"], result["walkers"] = len(vehicles) - 1, len(walkers)

        out = args.out / town
        out.mkdir(parents=True, exist_ok=True)
        saved, tick_times = 0, []
        for tick in range(args.warmup + args.frames):
            t = time.time()
            world.tick()
            rgb, inst, depth = (q.get(timeout=600.0) for q in queues)
            tick_times.append(time.time() - t)
            if len({rgb.frame, inst.frame, depth.frame}) != 1:
                raise RuntimeError("sensors out of sync")
            if tick % 10 == 0:
                peaks.sample()
            if tick < args.warmup:
                continue
            index = tick - args.warmup
            cv2.imwrite(str(out / f"{index:06d}_rgb.jpg"), to_bgr(rgb))
            cv2.imwrite(str(out / f"{index:06d}_instance.png"), to_bgr(inst))
            cv2.imwrite(str(out / f"{index:06d}_depth.png"), to_bgr(depth))
            saved += 1
        last_rgb, last_inst = to_bgr(rgb), to_bgr(inst)
        result["frames_saved"] = saved
        result["sim_fps"] = round(1 / float(np.mean(tick_times[args.warmup:])), 1)
        result["rgb_mean"] = round(float(last_rgb.mean()), 1)  # ~0 means black frames (no real rendering)
        ids = last_inst[:, :, 1].astype(np.int32) + 256 * last_inst[:, :, 0].astype(np.int32)  # BGR: G + B*256
        result["instance_ids_last_frame"] = int(len(np.unique(ids)))
        result["ok"] = saved == args.frames and result["rgb_mean"] > 5
    finally:
        for camera in cameras:
            camera.stop()
        for controller in controllers:
            controller.stop()
        for vehicle in vehicles:
            vehicle.set_autopilot(False, 8000)
        world.tick()
        client.apply_batch_sync([carla.command.DestroyActor(a) for a in [*cameras, *controllers, *walkers, *vehicles]])
        world.tick()
        tm.set_synchronous_mode(False)
        settings.synchronous_mode = False
        world.apply_settings(settings)
    return result


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--towns", nargs="+", default=["Town10HD_Opt", "Town03"])
    parser.add_argument("--vehicles", type=int, default=40)
    parser.add_argument("--walkers", type=int, default=60)
    parser.add_argument("--warmup", type=int, default=40)
    parser.add_argument("--frames", type=int, default=100)
    parser.add_argument("--out", type=Path, required=True)
    args = parser.parse_args()

    client = carla.Client("localhost", 2000)
    client.set_timeout(600.0)
    results = {"server_version": client.get_server_version(), "towns": []}
    for town in args.towns:
        peaks = Peaks()
        print(f"--- {town}", flush=True)
        try:
            result = run_town(client, town, args, peaks)
        except Exception as exc:  # report and carry on: the point is to see what breaks
            result = {"town": town, "ok": False, "error": f"{type(exc).__name__}: {exc}"}
        result.update(
            peak_ram_used_gb=round(peaks.ram_used_gb, 1),
            peak_carla_rss_gb=round(peaks.carla_rss_gb, 1),
            peak_vram_mb=int(peaks.vram_mb),
            server_alive_after=server_alive(),
        )
        print(json.dumps(result), flush=True)
        results["towns"].append(result)
        if not result["server_alive_after"]:
            print("CARLA server died; stopping")
            break
    (args.out / "carla_results.json").write_text(json.dumps(results, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
if REPORT.get("server_started"):
    sh(f"{PY} /tmp/carla_test.py --out {OUT}", check=False)
else:
    print("skipped: server not running")

## Verdict

In [ ]:
results_path = f"{OUT}/carla_results.json"
if os.path.exists(results_path):
    REPORT["recording"] = json.load(open(results_path))
towns = REPORT.get("recording", {}).get("towns", [])
REPORT["verdict"] = (
    "PASS" if REPORT.get("vulkan_ok") and len(towns) == 2 and all(t.get("ok") for t in towns) else "FAIL"
)
with open(f"{OUT}/report.json", "w") as f:
    json.dump(REPORT, f, indent=2)
if REPORT["verdict"] == "FAIL":
    sh(f"tail -n 40 {WORK}/carla_server.log", check=False)

try:
    import cv2
    import matplotlib.pyplot as plt

    shots = [(t["town"], f"{OUT}/{t['town']}/000099_rgb.jpg") for t in towns]
    shots = [(name, path) for name, path in shots if os.path.exists(path)]
    if shots:
        fig, axes = plt.subplots(1, len(shots), figsize=(7 * len(shots), 5), squeeze=False)
        for ax, (name, path) in zip(axes[0], shots, strict=True):
            ax.imshow(cv2.imread(path)[:, :, ::-1])
            ax.set_title(name)
            ax.axis("off")
        plt.show()
except Exception as exc:
    print("no preview:", exc)

sh("pkill -f CarlaUE4 || true", check=False, quiet=True)
print(f"\nVERDICT: {REPORT['verdict']}\n")
print(json.dumps(REPORT, indent=2))